# Demo M3/M4 — Training + Evaluation (số thật Tier B)

Notebook demo cho **M3 (training loop + TensorBoard)** và **M4 (evaluation)** theo `đề.txt` (E2, E3, O9, O10, O11).
Số liệu lấy từ run thật `results/runs/20261029-212336/resnet18-lr0.001-bs32-0/` (ResNet18 freeze, ảnh 64×64, 5 epoch, CPU).
Chạy: `jupyter notebook notebooks/20261029_train_eval_demo.ipynb` — các cell code đọc artifact có sẵn, không cần train lại.

## 1. Lịch sử train (M3 — E2/E3/O9/O10)

| epoch | train loss/acc | val loss/acc | → best |
|-------|---------------|--------------|--------|
| 1 | 1.4501 / 0.4993 | 1.3911 / 0.5288 | *best* |
| 2 | 1.3327 / 0.5416 | 1.3851 / 0.5336 | *best* |
| 3 | 1.3248 / 0.5460 | 1.3150 / 0.5628 | *best* |
| 4 | 1.3144 / 0.5486 | 1.2952 / 0.5654 | *best* |
| 5 | 1.3099 / 0.5505 | 1.3531 / 0.5554 | val tăng nhẹ — best vẫn epoch 4 |

Loop 1 epoch (bám slide 8–9 Chương 3): `zero_grad → forward → loss → backward → step`. Checkpoint `best.pt` (44.8 MB) lưu theo val acc. TensorBoard tags: `train/loss`, `train/accuracy`, `val/loss`, `val/accuracy`, `lr`, `model/grad_norm`, `samples/train_images`.

In [ ]:
import json
from pathlib import Path

run = Path('results/runs/20261029-212336/resnet18-lr0.001-bs32-0')
train_m = json.loads((run / 'metrics.json').read_text())
eval_m = json.loads((run / 'eval' / 'metrics.json').read_text())

print('best val acc:', train_m['train_summary']['best_val_accuracy'])
print('test acc     :', eval_m['test']['accuracy'])
print('macro-F1     :', eval_m['test']['macro_f1'])

for i, (tl, ta, vl, va) in enumerate(zip(train_m['history']['train_loss'], train_m['history']['train_acc'], train_m['history']['val_loss'], train_m['history']['val_acc']), 1):
    print(f'epoch {i}: train {tl:.4f}/{ta:.4f} | val {vl:.4f}/{va:.4f}')

## 2. Vẽ lại đường train/val từ history (M3 — TensorBoard offline)

Tương đương các scalar trong TensorBoard (`tb/` của run). Dùng khi không mở được TensorBoard.

In [ ]:
import matplotlib.pyplot as plt

h = train_m['history']
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h['train_loss'], label='train'); ax[0].plot(h['val_loss'], label='val')
ax[0].set(title='Loss', xlabel='epoch', ylabel='loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(h['train_acc'], label='train'); ax[1].plot(h['val_acc'], label='val')
ax[1].set(title='Accuracy', xlabel='epoch', ylabel='acc'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Kết quả test (M4 — O11)

| metric | giá trị |
|--------|----------|
| test accuracy | **0.5582** |
| macro-F1 | **0.5535** |
| inference | 4.31 ms/mẫu (CPU) |

Per-class F1 thấp nhất: **cat 0.442 / deer 0.424** (cặp dễ nhầm cổ điển CIFAR-10); cao nhất: automobile 0.667 / ship 0.622. Tổng confusion matrix = 10.000 = số mẫu test.

In [ ]:
per = eval_m['test']['per_class']
for c in sorted(per, key=int):
    d = per[c]
    print(f"lop {c}: P={d['precision']:.3f} R={d['recall']:.3f} F1={d['f1']:.3f} (n={d['support']})")

## 4. Ảnh confusion matrix + per-class F1 (M4 — đã sinh trong `eval/`)

`eval/confusion_matrix.png` và `eval/per_class_f1.png` — xem trực tiếp trong thư mục run.

In [ ]:
from IPython.display import Image
Image(str(run / 'eval' / 'confusion_matrix.png'), width=600)